In [1]:
import sys
import numpy as np
import pandas as pd
import sklearn, matplotlib, seaborn

**Get dataset filepath**

In [2]:
from pathlib import Path

# Run this notebook from the project root directory.
#
# Required project structure:
#
# project_root/
# ├── dataset/
# │   ├── kddcup.data_10_percent_corrected
# │   ├── ...
# └── data_preprocessing.ipynb
#
# The 'dataset' folder must be located in the same directory, and all dataset files must keep their original filenames.

PROJECT_ROOT = Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "dataset"

TRAIN_FILE = DATA_DIR / "kddcup.data_10_percent_corrected"
TEST_FILE = DATA_DIR / "corrected"
NAMES_FILE = DATA_DIR / "kddcup.names.txt"
ATTACK_TYPES_FILE = DATA_DIR / "training_attack_types.txt"

In [3]:
PROJECT_ROOT, DATA_DIR

(WindowsPath('C:/Users/Gzq/Desktop/deep learning project'),
 WindowsPath('C:/Users/Gzq/Desktop/deep learning project/dataset'))

**Read kddcup.names.txt, Analyze data structure**

In [4]:
# Read the official KDD Cup 1999 feature-definition file.
with NAMES_FILE.open("r", encoding="utf-8") as file:
    lines = [line.strip() for line in file if line.strip()]

# all traffic-class labels (smurf, neptune, back, ...)
declared_labels = [label.strip() for label in lines[0].rstrip(".").split(",")]

# The remaining lines' format: feature_name: type
feature_schema = []

for line in lines[1:]:
    cleaned_line = line.rstrip(".")

    feature_name, feature_type = cleaned_line.split(":", maxsplit=1)

    feature_schema.append({
        "feature": feature_name.strip(),
        "declared_type": feature_type.strip(),
    })

schema_df = pd.DataFrame(feature_schema)

feature_names = schema_df["feature"].tolist()
column_names = feature_names + ["label"]

if schema_df["feature"].duplicated().any():
    print("duplicate features")


In [5]:
len(declared_labels), len(feature_names), len(column_names)

(23, 41, 42)

In [6]:
schema_df["declared_type"].value_counts()

declared_type
continuous    34
symbolic       7
Name: count, dtype: int64

In [7]:
schema_df.loc[schema_df["declared_type"] == "symbolic","feature"].tolist()

['protocol_type',
 'service',
 'flag',
 'land',
 'logged_in',
 'is_host_login',
 'is_guest_login']

In [8]:
schema_df

,feature,declared_type
0,duration,continuous
1,protocol_type,symbolic
2,service,symbolic
3,flag,symbolic
4,src_bytes,continuous
5,dst_bytes,continuous
6,land,symbolic
7,wrong_fragment,continuous
8,urgent,continuous
9,hot,continuous


**raw data integrity checking**

In [9]:
import csv
from collections import Counter

for dataset_name, file_path in {
    "Training data": TRAIN_FILE,
    "Final test data": TEST_FILE,
}.items():

    row_length_counts = Counter()
    row_count = blank_row_count = 0

    with file_path.open("r", encoding="utf-8", newline="") as file:
        for row in csv.reader(file):
            if not row:
                blank_row_count += 1
                continue

            row_count += 1
            row_length_counts[len(row)] += 1

    print(f"\n{dataset_name}: \n")
    print(f"Rows: {row_count:,}")
    print(f"Blank rows: {blank_row_count}")
    print(f"Column-count distribution: {dict(row_length_counts)}")

    assert row_length_counts == Counter({42: row_count}), (
        f"{dataset_name} contains malformed rows."
    )


Training data: 

Rows: 494,021
Blank rows: 0
Column-count distribution: {42: 494021}

Final test data: 

Rows: 311,029
Blank rows: 0
Column-count distribution: {42: 311029}


**Data missing and illegal value checking**

In [10]:
train_df = pd.read_csv(TRAIN_FILE, names=column_names, header=None, low_memory=False)
string_columns = train_df.select_dtypes(include=["str"]).columns.tolist()
numeric_columns = train_df.select_dtypes(include=[np.number]).columns.tolist()

missing_counts = train_df.isna().sum()
missing_counts = missing_counts[missing_counts > 0]

blank_string_counts = pd.Series(
    {
        column: train_df[column].str.strip().eq("").sum()
        for column in string_columns
    },
    dtype="int64"
)
blank_string_counts = blank_string_counts[blank_string_counts > 0]

numeric_values = train_df[numeric_columns].to_numpy()
non_finite_counts = pd.Series(
    (~np.isfinite(numeric_values)).sum(axis=0),
    index=numeric_columns
)
non_finite_counts = non_finite_counts[non_finite_counts > 0]

In [11]:
string_columns, len(numeric_columns)

(['protocol_type', 'service', 'flag', 'label'], 38)

In [12]:
#missing values
print("Total missing values:", int(train_df.isna().sum().sum()))

Total missing values: 0


In [13]:
# Columns containing missing values
print(missing_counts if not missing_counts.empty else "None")

None


In [14]:
#Blank strings
int(blank_string_counts.sum())

0

In [15]:
# Columns containing blank strings
print(blank_string_counts if not blank_string_counts.empty else "None")

None


In [16]:
# Total NaN or infinite values
int(non_finite_counts.sum())

0

**String format check**

In [17]:
string_format_summary = []

for column in string_columns:
    stripped = train_df[column].str.strip()

    string_format_summary.append({
        "column": column,
        "unique_raw": train_df[column].nunique(),
        "unique_after_strip": stripped.nunique(),
        "rows_with_outer_whitespace": int(train_df[column].ne(stripped).sum()),
    })

display(pd.DataFrame(string_format_summary))

labels_ending_with_period = train_df["label"].str.endswith(".").sum()

print("Labels ending with '.':",f"{labels_ending_with_period:,} / {len(train_df):,}")

,column,unique_raw,unique_after_strip,rows_with_outer_whitespace
0,protocol_type,3,3,0
1,service,66,66,0
2,flag,11,11,0
3,label,23,23,0


Labels ending with '.': 494,021 / 494,021


In [18]:
# protocol_type values
train_df["protocol_type"].unique()

<StringArray>
['tcp', 'udp', 'icmp']
Length: 3, dtype: str

In [19]:
# flag values
train_df["flag"].unique()

<StringArray>
['SF', 'S1', 'REJ', 'S2', 'S0', 'S3', 'RSTO', 'RSTR', 'RSTOS0', 'OTH', 'SH']
Length: 11, dtype: str

In [20]:
# Raw label values
train_df["label"].unique()

<StringArray>
[         'normal.', 'buffer_overflow.',      'loadmodule.',
            'perl.',         'neptune.',           'smurf.',
    'guess_passwd.',             'pod.',        'teardrop.',
       'portsweep.',         'ipsweep.',            'land.',
       'ftp_write.',            'back.',            'imap.',
           'satan.',             'phf.',            'nmap.',
        'multihop.',     'warezmaster.',     'warezclient.',
             'spy.',         'rootkit.']
Length: 23, dtype: str

**Remove trailing spaces and periods**

In [21]:
train_df["label"] = (train_df["label"].str.strip().str.removesuffix("."))

**Create a binary classification target**

In [22]:
train_df["is_anomaly"] = (train_df["label"].ne("normal")).astype("int8")

**View the binary classification target data**

In [23]:
binary_counts = (train_df["is_anomaly"].value_counts().sort_index().rename(index={0: "Normal", 1: "Anomaly"}))
binary_percentages = (train_df["is_anomaly"].value_counts(normalize=True).sort_index().mul(100).rename(index={0: "Normal", 1: "Anomaly"}))

binary_summary = pd.DataFrame({
    "count": binary_counts,
    "percentage": binary_percentages
})

display(binary_summary)

,count,percentage
is_anomaly,,
Normal,97278,19.691066
Anomaly,396743,80.308934


**Grouping 22 specific attack labels into 4 major categories makes it easier to evaluate the model based on attack types in the future (as the distribution of attack types is uneven)**

In [24]:
attack_category_map = {"normal": "Normal"}

category_name_map = {
    "dos": "DoS",
    "probe": "Probe",
    "r2l": "R2L",
    "u2r": "U2R",
}

with ATTACK_TYPES_FILE.open("r", encoding="utf-8") as file:
    for line in file:
        line = line.strip()

        if not line:
            continue

        attack_name, category = line.split()
        attack_category_map[attack_name] = category_name_map[category.lower()]


category_order = ["Normal", "DoS", "Probe", "R2L", "U2R"]

train_df["attack_category"] = pd.Categorical(
    train_df["label"].map(attack_category_map),
    categories=category_order,
    ordered=False
)

category_counts = (train_df["attack_category"].value_counts().reindex(category_order))
category_percentages = category_counts / len(train_df) * 100

category_summary = pd.DataFrame({"count": category_counts, "percentage": category_percentages})

display(category_summary)

,count,percentage
attack_category,,
Normal,97278,19.691066
DoS,391458,79.239142
Probe,4107,0.831341
R2L,1126,0.227926
U2R,52,0.010526


In [25]:
# Unmapped category rows
train_df["attack_category"].isna().sum()

np.int64(0)

**label count**

In [26]:
label_counts = train_df["label"].value_counts()

label_summary = (label_counts.rename("count").to_frame().reset_index().rename(columns={"index": "label"}))
label_summary["attack_category"] = (label_summary["label"].map(attack_category_map))
label_summary["percentage"] = (label_summary["count"] / len(train_df) * 100)
label_summary = label_summary[["label", "attack_category", "count", "percentage"]]

display(label_summary)

,label,attack_category,count,percentage
0,smurf,DoS,280790,56.837665
1,neptune,DoS,107201,21.699685
2,normal,Normal,97278,19.691066
3,back,DoS,2203,0.445932
4,satan,Probe,1589,0.321646
5,ipsweep,Probe,1247,0.252418
6,portsweep,Probe,1040,0.210517
7,warezclient,R2L,1020,0.206469
8,teardrop,DoS,979,0.198170
9,pod,DoS,264,0.053439


In [27]:
# labels that are less than 100
rare_labels = label_summary.loc[label_summary["count"] < 100,["label", "attack_category", "count"]]
display(rare_labels)

,label,attack_category,count
11,guess_passwd,R2L,53
12,buffer_overflow,U2R,30
13,land,DoS,21
14,warezmaster,R2L,20
15,imap,R2L,12
16,rootkit,U2R,10
17,loadmodule,U2R,9
18,ftp_write,R2L,8
19,multihop,R2L,7
20,phf,R2L,4


**Analyze completely identical records**

In [28]:
# Determine the columns used for identifying duplicates
duplicate_check_columns = feature_names + ["label"]

redundant_duplicate_mask = train_df.duplicated(subset=duplicate_check_columns,keep="first")
duplicate_group_mask = train_df.duplicated(subset=duplicate_check_columns, keep=False)

duplicate_by_category = (
    train_df.assign(is_redundant_duplicate=redundant_duplicate_mask).groupby("attack_category", observed=True)
    .agg(total_rows=("is_redundant_duplicate", "size"),redundant_duplicates=("is_redundant_duplicate", "sum")))

display(duplicate_by_category)

,total_rows,redundant_duplicates
attack_category,,
Normal,97278,9446
DoS,391458,336886
Probe,4107,1976
R2L,1126,127
U2R,52,0


In [29]:
# Total rows
len(train_df)

494021

In [30]:
# Unique rows
int(train_df[duplicate_check_columns].drop_duplicates().shape[0])

145586

In [31]:
# Redundant duplicate rows
int(redundant_duplicate_mask.sum())

348435

**Check that the input features are exactly the same but the labels are different**

In [32]:
# group by features
feature_groups = train_df.groupby(feature_names, sort=False, dropna=False)

labels_per_feature_group = feature_groups["label"].nunique()
binary_targets_per_feature_group = feature_groups["is_anomaly"].nunique()

multiclass_conflicts = labels_per_feature_group[labels_per_feature_group > 1]
binary_conflicts = binary_targets_per_feature_group[binary_targets_per_feature_group > 1]


In [33]:
# The number of record with the same characteristics but different attack types
len(multiclass_conflicts)

2

In [34]:
"""
There are records with the same characteristics, but some are regarded as abnormal while others are considered normal. The number of these records
"""
len(binary_conflicts)

1

**Display Conflicts**

In [35]:
conflicting_feature_keys = (multiclass_conflicts.index.to_frame(index=False))

conflicting_rows = (train_df.reset_index(names="original_row").merge(conflicting_feature_keys, on=feature_names, how="inner"))

# pick up some features
columns_to_show = ["original_row", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", "srv_count", "label", "attack_category",]

display(conflicting_rows[columns_to_show].sort_values(["service", "src_bytes", "label", "original_row"]))

,original_row,protocol_type,service,flag,src_bytes,dst_bytes,srv_count,label,attack_category
1,148773,icmp,ecr_i,SF,8,0,1,ipsweep,Probe
2,345835,icmp,ecr_i,SF,8,0,1,portsweep,Probe
0,143854,icmp,tim_i,SF,564,0,1,normal,Normal
3,345951,icmp,tim_i,SF,564,0,1,pod,DoS


**Remove duplicate records**

In [36]:
conflict_keys = multiclass_conflicts.index.to_frame(index=False)
conflict_keys["_has_label_conflict"] = True

conflict_marker = (
    train_df[feature_names]
    .merge(
        conflict_keys,
        on=feature_names,
        how="left"
    )["_has_label_conflict"]
    .fillna(False)
    .astype(bool)
    .to_numpy()
)

# Create a dedicated dataframe for model development
model_df = (train_df.loc[~conflict_marker].drop_duplicates(subset=duplicate_check_columns, keep="first").copy())


print(f"Removed rows: {conflict_marker.sum():,}")
print(f"Model development rows: {len(model_df):,}")
print(f"Total removed:", f"{len(train_df) - len(model_df):,}")

Removed rows: 4
Model development rows: 145,582
Total removed: 348,439


**Divide the clean data into normal traffic pool and attack traffic pool**

In [37]:
normal_pool_df = (model_df.loc[model_df["is_anomaly"] == 0].copy())
anomaly_reference_df = (model_df.loc[model_df["is_anomaly"] == 1].copy())

print(f"Normal records: {len(normal_pool_df):,}")
print(f"Anomaly reference records: {len(anomaly_reference_df):,}")

Normal records: 87,831
Anomaly reference records: 57,751


**Split tranining set, evaluation set and test set**

In [38]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

# 70% training, 30% validation and test set
normal_train_df, normal_temp_df = train_test_split(normal_pool_df, test_size=0.30, random_state=RANDOM_STATE, shuffle=True)

# second split, 315% validation and 15% test
normal_validation_df, normal_calibration_df = train_test_split(normal_temp_df, test_size=0.50, random_state=RANDOM_STATE, shuffle=True)

# show datasets data
split_summary = pd.DataFrame({
    "rows": [
        len(normal_train_df),
        len(normal_validation_df),
        len(normal_calibration_df)
    ],
    "percentage": [
        len(normal_train_df) / len(normal_pool_df) * 100,
        len(normal_validation_df) / len(normal_pool_df) * 100,
        len(normal_calibration_df) / len(normal_pool_df) * 100
    ]
}, index=[
    "Normal Train",
    "Normal Validation",
    "Normal Calibration"
])

display(split_summary)

,rows,percentage
Normal Train,61481,69.999203
Normal Validation,13175,15.000398
Normal Calibration,13175,15.000398


**Feature distribution check**

In [39]:
from pandas.api.types import is_numeric_dtype

feature_profile_rows = []

for feature in feature_names:
    series = normal_train_df[feature]
    value_counts = series.value_counts(dropna=False)

    most_common_value = value_counts.index[0]
    most_common_count = int(value_counts.iloc[0])

    profile = {
        "feature": feature,
        "dtype": str(series.dtype),
        "unique_values": int(series.nunique(dropna=False)),
        "most_common_value": most_common_value,
        "most_common_percentage": (most_common_count / len(series) * 100),
    }

    
    if is_numeric_dtype(series): # if it is a numerical feature, then record the minimum and maximum values separately
        profile["minimum"] = series.min()
        profile["maximum"] = series.max()
    else:
        profile["minimum"] = np.nan
        profile["maximum"] = np.nan

    feature_profile_rows.append(profile)

feature_profile_df = pd.DataFrame(feature_profile_rows)
constant_features = feature_profile_df.loc[feature_profile_df["unique_values"] == 1].copy()


print("Constant features:")
display(constant_features)

Constant features:


,feature,dtype,unique_values,most_common_value,most_common_percentage,minimum,maximum
6,land,int64,1,0,100.0,0.0,0.0
7,wrong_fragment,int64,1,0,100.0,0.0,0.0
8,urgent,int64,1,0,100.0,0.0,0.0
19,num_outbound_cmds,int64,1,0,100.0,0.0,0.0
20,is_host_login,int64,1,0,100.0,0.0,0.0


**Check whether the feature is constant in the normal training process, and then determine if it remains constant in the attack data**

In [40]:
constant_feature_names = constant_features["feature"].tolist()

comparison_rows = []

datasets_to_compare = {
    "Training set": normal_train_df,
    "Anomalies": anomaly_reference_df,
    "Oiginalt data": model_df,
}

for dataset_name, dataframe in datasets_to_compare.items():
    for feature in constant_feature_names:
        series = dataframe[feature]

        comparison_rows.append({
            "dataset": dataset_name,
            "feature": feature,
            "unique_values": series.nunique(),
            "minimum": series.min(),
            "maximum": series.max(),
            "nonzero_rows": int(series.ne(0).sum()),
        })

constant_feature_comparison = pd.DataFrame(comparison_rows)

display(constant_feature_comparison.sort_values(["feature", "dataset"]))

,dataset,feature,unique_values,minimum,maximum,nonzero_rows
9,Anomalies,is_host_login,1,0,0,0
14,Oiginalt data,is_host_login,1,0,0,0
4,Training set,is_host_login,1,0,0,0
5,Anomalies,land,2,0,1,19
10,Oiginalt data,land,2,0,1,20
0,Training set,land,1,0,0,0
8,Anomalies,num_outbound_cmds,1,0,0,0
13,Oiginalt data,num_outbound_cmds,1,0,0,0
3,Training set,num_outbound_cmds,1,0,0,0
7,Anomalies,urgent,3,0,2,3


In [41]:
numeric_feature_names = [feature for feature in feature_names if is_numeric_dtype(normal_train_df[feature])]

normal_numeric = normal_train_df[numeric_feature_names]
numeric_summary = normal_numeric.quantile([0.00, 0.50, 0.90, 0.99, 0.999, 1.00]).T # calculate the quantiles of each numerical feature

numeric_summary.columns = ["minimum", "median", "p90", "p99", "p99_9", "maximum",]

numeric_summary["mean"] = normal_numeric.mean()
numeric_summary["std"] = normal_numeric.std()
numeric_summary["skewness"] = normal_numeric.skew()
numeric_summary["zero_percentage"] = (normal_numeric.eq(0).mean() * 100)

log_transform_candidates = numeric_summary.loc[
    (numeric_summary["minimum"] >= 0)
    & (numeric_summary["maximum"] > 10)
    & (numeric_summary["skewness"] > 2)].sort_values("skewness", ascending=False)

print("Number of numeric features:", len(numeric_feature_names))
display(log_transform_candidates)

Number of numeric features: 38


,minimum,median,p90,p99,p99_9,maximum,mean,std,skewness,zero_percentage
num_root,0.0,0.0,0.0,0.0,9.00,993.0,0.074804,5.689126,148.344514,99.364031
num_compromised,0.0,0.0,0.0,0.0,0.00,884.0,0.043900,5.089010,148.022305,99.917048
num_file_creations,0.0,0.0,0.0,0.0,1.00,22.0,0.004343,0.151754,93.731970,99.726745
dst_bytes,0.0,614.0,7609.0,33730.0,239398.76,5134218.0,3758.269628,41198.360618,70.377222,12.763293
src_bytes,0.0,240.0,971.0,9502.4,63318.08,2194619.0,1246.295311,35786.565374,60.098572,5.385404
hot,0.0,0.0,0.0,0.0,18.00,30.0,0.047071,0.876255,24.925522,99.403068
duration,0.0,0.0,1.0,5996.8,17638.04,27790.0,190.582635,1305.718905,9.943084,88.671297
count,0.0,4.0,20.0,69.0,259.08,511.0,8.882663,18.492420,9.615780,0.001627
srv_count,0.0,5.0,27.0,125.0,271.04,510.0,11.967323,22.805819,6.899231,0.001627


1. **Skewed Feature Distribution**  
   Some features contain a large number of zeros or small values, while a few samples have extremely large values.

2. **Limitation of Min-Max Scaling**  
   Direct Min-Max scaling may compress most normal samples into a narrow region near 0 due to the presence of extreme values.

3. **Benefit of Log Transformation**  
   Applying a logarithmic transformation preserves relative magnitude differences while reducing the impact of extreme values.

**Examine the new features outside the training set**

In [42]:
categorical_feature_names = ["protocol_type", "service", "flag",] # The column of categories that requires one-hot encoding

comparison_datasets = {
    "Normal Validation": normal_validation_df,
    "Normal Calibration": normal_calibration_df,
    "Anomaly Reference": anomaly_reference_df,
}


# Extract the categories that have been encountered in the normal training set.
for feature in categorical_feature_names:
    training_categories = set(normal_train_df[feature].unique())

    print(f"\nFeature: {feature}")
    print("-" * 60)
    print(
        f"Categories in training set: "
        f"{len(training_categories)}"
    )

    for dataset_name, dataframe in comparison_datasets.items():
        dataset_categories = set(dataframe[feature].unique())
        unseen_categories = sorted(
            dataset_categories - training_categories
        )

        print(
            f"{dataset_name}: "
            f"{len(dataset_categories)} categories, "
            f"{len(unseen_categories)} unseen"
        )

        if unseen_categories:
            print("  Unseen values:", unseen_categories)


Feature: protocol_type
------------------------------------------------------------
Categories in training set: 3
Normal Validation: 3 categories, 0 unseen
Normal Calibration: 3 categories, 0 unseen
Anomaly Reference: 3 categories, 0 unseen

Feature: service
------------------------------------------------------------
Categories in training set: 25
Normal Validation: 18 categories, 0 unseen
Normal Calibration: 19 categories, 0 unseen
Anomaly Reference: 62 categories, 41 unseen
  Unseen values: ['Z39_50', 'bgp', 'courier', 'csnet_ns', 'ctf', 'daytime', 'discard', 'echo', 'efs', 'exec', 'gopher', 'hostnames', 'http_443', 'imap4', 'iso_tsap', 'klogin', 'kshell', 'ldap', 'link', 'login', 'mtp', 'name', 'netbios_dgm', 'netbios_ns', 'netbios_ssn', 'netstat', 'nnsp', 'nntp', 'pm_dump', 'pop_2', 'printer', 'remote_job', 'rje', 'sql_net', 'sunrpc', 'supdup', 'systat', 'uucp', 'uucp_path', 'vmnet', 'whois']

Feature: flag
------------------------------------------------------------
Categories i

**Some classification values appear outside the training set, especially in the attack records. Therefore, one-hot encoding is used for the regular training set, and set handle_unknown="ignore" to ensure that unseen categories can be safely converted**

**Check how many records were affected by the unseen categories**

In [43]:
for dataset_name, dataframe in comparison_datasets.items():
    any_unseen_mask = pd.Series(False, index=dataframe.index)

    print(f"\n{dataset_name}")
    print("-" * 60)

    for feature in categorical_feature_names:
        known_values = set(normal_train_df[feature].unique())

        unseen_mask = ~dataframe[feature].isin(known_values)
        any_unseen_mask |= unseen_mask

        print(
            f"{feature}: "
            f"{unseen_mask.sum():,} unseen rows "
            f"({unseen_mask.mean() * 100:.4f}%)"
        )


Normal Validation
------------------------------------------------------------
protocol_type: 0 unseen rows (0.0000%)
service: 0 unseen rows (0.0000%)
flag: 0 unseen rows (0.0000%)

Normal Calibration
------------------------------------------------------------
protocol_type: 0 unseen rows (0.0000%)
service: 0 unseen rows (0.0000%)
flag: 1 unseen rows (0.0076%)

Anomaly Reference
------------------------------------------------------------
protocol_type: 0 unseen rows (0.0000%)
service: 4,242 unseen rows (7.3453%)
flag: 51 unseen rows (0.0883%)


**Use different encoding methods for different characteristics**

In [44]:
categorical_feature_names = ["protocol_type", "service", "flag",] # string category feature

# Features with obvious right skewness and extreme values
log_transform_features = ["duration", "src_bytes", "dst_bytes", "hot", "num_compromised", "num_root", "num_file_creations", "count", "srv_count",]

other_numeric_features = [feature for feature in feature_names if feature not in categorical_feature_names and feature not in log_transform_features]

In [45]:
categorical_feature_names

['protocol_type', 'service', 'flag']

In [46]:
log_transform_features

['duration',
 'src_bytes',
 'dst_bytes',
 'hot',
 'num_compromised',
 'num_root',
 'num_file_creations',
 'count',
 'srv_count']

In [47]:
other_numeric_features

['land',
 'wrong_fragment',
 'urgent',
 'num_failed_logins',
 'logged_in',
 'root_shell',
 'su_attempted',
 'num_shells',
 'num_access_files',
 'num_outbound_cmds',
 'is_host_login',
 'is_guest_login',
 'serror_rate',
 'srv_serror_rate',
 'rerror_rate',
 'srv_rerror_rate',
 'same_srv_rate',
 'diff_srv_rate',
 'srv_diff_host_rate',
 'dst_host_count',
 'dst_host_srv_count',
 'dst_host_same_srv_rate',
 'dst_host_diff_srv_rate',
 'dst_host_same_src_port_rate',
 'dst_host_srv_diff_host_rate',
 'dst_host_serror_rate',
 'dst_host_srv_serror_rate',
 'dst_host_rerror_rate',
 'dst_host_srv_rerror_rate']

**1. categorical_feature_names**
- Categorical features represented as strings will be processed using **One-Hot Encoding**

**2. Log transform features**
- Non-negative numerical features with highly right-skewed distributions will be processed using:
  **log1p()** transformation and then **Min-Max Scaling**

**3. Other numeric features**
- All other numerical features will be processed using **Min-Max Scaling** only

**Convert the original 41 features into a 74-dimensional numerical matrix that the autoencoder can directly use.**

In [48]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, MinMaxScaler, OneHotEncoder

#construct pipeline
log_numeric_pipeline = Pipeline([
    ("log1p", FunctionTransformer(np.log1p, validate=False, feature_names_out="one-to-one")),
    ("minmax",MinMaxScaler(feature_range=(0, 1), clip=True)),
])


preprocessor = ColumnTransformer(
    transformers=[
        ("log_numeric", log_numeric_pipeline, log_transform_features),
        ("other_numeric", MinMaxScaler(feature_range=(0, 1), clip=True), other_numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32), categorical_feature_names),
    ],
    remainder="drop", sparse_threshold=0, verbose_feature_names_out=True,
)

# Fit only on the normal training data
X_train_normal = preprocessor.fit_transform(normal_train_df[feature_names]).astype(np.float32)

# validation and test set only use transform
X_validation_normal = preprocessor.transform(normal_validation_df[feature_names]).astype(np.float32)
X_calibration_normal = preprocessor.transform(normal_calibration_df[feature_names]).astype(np.float32)

processed_feature_names = (preprocessor.get_feature_names_out())

In [49]:
X_train_normal.shape, X_validation_normal.shape, X_calibration_normal.shape

((61481, 74), (13175, 74), (13175, 74))

In [50]:
# output dtype and length
X_train_normal.dtype, len(processed_feature_names)

(dtype('float32'), 74)

**Confirm that the output structure is correct**

In [51]:
# get one-hot encoder
fitted_encoder = (preprocessor.named_transformers_["categorical"])

# Check how many one-hot dimensions are generated for each categorical feature
category_output_counts = {feature: len(categories) for feature, categories in zip(categorical_feature_names, fitted_encoder.categories_)}

In [52]:
# output dimensions, total categorical dimensions
category_output_counts, sum(category_output_counts.values())

({'protocol_type': 3, 'service': 25, 'flag': 8}, 36)

In [53]:
# Find the flag that was not encountered during the training phase in the Calibration process
known_flags = set(fitted_encoder.categories_[categorical_feature_names.index("flag")])

unknown_flag_mask = (~normal_calibration_df["flag"].isin(known_flags))
unknown_positions = np.flatnonzero(unknown_flag_mask.to_numpy())

print(normal_calibration_df.loc[unknown_flag_mask, ["protocol_type", "service", "flag"]])

      protocol_type   service flag
41848           tcp  ftp_data  OTH


In [54]:
# Determine the column range corresponding to the flag one-hot encoding in the final matrix
categorical_slice = (preprocessor.output_indices_["categorical"])

#  find the position of "flag" within the list of categorical features
flag_position = categorical_feature_names.index("flag")

# count how many one-hot columns appear before the flag block inside the categorical section
flag_local_start = sum(len(categories) for categories in fitted_encoder.categories_[:flag_position])

# the number of flag categories observed during fitting on the normal training split
flag_width = len(fitted_encoder.categories_[flag_position])

flag_global_start = (categorical_slice.start + flag_local_start)
flag_global_end = flag_global_start + flag_width

for position in unknown_positions:
    flag_block = X_calibration_normal[position, flag_global_start:flag_global_end]

    print(
        f"Matrix row {position}, "
        f"flag block sum = {flag_block.sum()}, "
        f"values = {flag_block}"
    )

Matrix row 147, flag block sum = 0.0, values = [0. 0. 0. 0. 0. 0. 0. 0.]


**Convert the attack reference set into a model input matrix in the same format as the normal data**

In [55]:
X_anomaly_reference = preprocessor.transform(anomaly_reference_df[feature_names]).astype(np.float32)
y_anomaly_reference = (anomaly_reference_df["is_anomaly"].to_numpy(dtype=np.int8))

In [56]:
# verify convert result
X_anomaly_reference.shape, y_anomaly_reference.shape, np.isfinite(X_anomaly_reference).all(), 

((57751, 74), (57751,), np.True_)

**After the preprocessing process, check which dimensions in the normal training matrix have zero variance**

In [57]:
train_feature_ranges = np.ptp(X_train_normal, axis=0)

zero_variance_indices = np.flatnonzero(train_feature_ranges == 0) # find the positions of all zero-variance columns
zero_variance_names = processed_feature_names[zero_variance_indices] # convert them to processed feature name

zero_variance_summary = []


# for each zero-variance processed feature, check the values of this dimension in the anomaly reference
for index, feature_name in zip(
    zero_variance_indices,
    zero_variance_names
):
    anomaly_values = X_anomaly_reference[:, index]

    zero_variance_summary.append({
        "processed_feature": feature_name,
        "train_value": X_train_normal[0, index],
        "anomaly_minimum": anomaly_values.min(),
        "anomaly_maximum": anomaly_values.max(),
        "anomaly_nonzero_rows": int(
            np.count_nonzero(anomaly_values)
        ),
        "anomaly_nonzero_percentage": (
            np.count_nonzero(anomaly_values)
            / len(anomaly_values)
            * 100
        ),
    })

zero_variance_summary_df = pd.DataFrame(zero_variance_summary)

display(zero_variance_summary_df)

,processed_feature,train_value,anomaly_minimum,anomaly_maximum,anomaly_nonzero_rows,anomaly_nonzero_percentage
0,other_numeric__land,0.0,0.0,1.0,19,0.032900
1,other_numeric__wrong_fragment,0.0,0.0,1.0,1121,1.941092
2,other_numeric__urgent,0.0,0.0,1.0,3,0.005195
3,other_numeric__num_outbound_cmds,0.0,0.0,0.0,0,0.000000
4,other_numeric__is_host_login,0.0,0.0,0.0,0,0.000000


**Five processed dimensions had zero variance in the normal training matrix. They were kept because several of them, especially wrong_fragment, become non-zero in anomaly records and may therefore increase reconstruction error for attacks. This supports retaining the full feature schema rather than removing constant-in-normal features**

### Rationale for Retaining All Features
In the preprocessing step, all 41 original input features were retained. Although in the normal training set division, some features remained constant or nearly constant, some of these features became non-zero values in abnormal samples, thus providing useful abnormal signals. Since the autoencoder learns the normal traffic patterns, deviations in these typically stable features may increase the reconstruction error. Retaining all features also helps maintain the original KDD Cup 1999 data structure and avoids inconsistencies between the preprocessing, saved artifacts, and the final test transformation.

### Save the processed data

In [58]:
import json
import joblib

ARTIFACT_DIR = (PROJECT_ROOT / "artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

data_file = ARTIFACT_DIR / "kdd99_preprocessed_data.npz"
preprocessor_file = ARTIFACT_DIR / "kdd99_preprocessor.joblib"
metadata_file = ARTIFACT_DIR / "kdd99_preprocessing_metadata.json"

# save the processed matrix, the evaluation labels, and the original row indices
np.savez_compressed(
    data_file,

    X_train_normal=X_train_normal,
    X_validation_normal=X_validation_normal,
    X_calibration_normal=X_calibration_normal,
    X_anomaly_reference=X_anomaly_reference,

    y_anomaly_reference=y_anomaly_reference,

    anomaly_labels=(anomaly_reference_df["label"].astype(str).to_numpy()),
    anomaly_categories=(anomaly_reference_df["attack_category"].astype(str).to_numpy()),

    train_original_indices=(normal_train_df.index.to_numpy(dtype=np.int64)),
    validation_original_indices=(normal_validation_df.index.to_numpy(dtype=np.int64)),
    calibration_original_indices=(normal_calibration_df.index.to_numpy(dtype=np.int64)),
    anomaly_original_indices=(anomaly_reference_df.index.to_numpy(dtype=np.int64)),
)

# save preprocessor
joblib.dump(preprocessor, preprocessor_file)

# preprocess instruction for my teammates
metadata = {
    "dataset": "KDD Cup 1999",
    "training_source": TRAIN_FILE.name,
    "final_test_source": TEST_FILE.name,
    "random_state": RANDOM_STATE,

    "original_feature_count": len(feature_names),
    "processed_feature_count": len(processed_feature_names),

    "original_feature_names": feature_names,
    "processed_feature_names": processed_feature_names.tolist(),

    "categorical_features": categorical_feature_names,
    "log_transform_features": log_transform_features,
    "other_numeric_features": other_numeric_features,

    "transformations": {
        "categorical": ("OneHotEncoder(handle_unknown='ignore')"),
        "log_numeric": ("log1p followed by MinMaxScaler"),
        "other_numeric": "MinMaxScaler",
        "numeric_range": [0, 1],
        "clip_out_of_range": True,
        "output_dtype": "float32",
    },

    "data_cleaning": {
        "exact_duplicates_removed": 348435,
        "conflicting_rows_removed": 4,
        "normal_records_after_cleaning": len(normal_pool_df),
        "anomaly_records_after_cleaning": len(anomaly_reference_df),
    },

    "splits": {
        "normal_train": len(X_train_normal),
        "normal_validation": len(X_validation_normal),
        "normal_calibration": len(X_calibration_normal),
        "anomaly_reference": len(X_anomaly_reference),
    },

    "data_roles": {
        "normal_train": "Autoencoder training only",
        "normal_validation": "Early stopping and model selection",
        "normal_calibration": "Anomaly-threshold calibration",
        "anomaly_reference": "Development evaluation only",
        "corrected": "Untouched final test set",
    },
}

# save files
with metadata_file.open("w", encoding="utf-8") as file:
    json.dump( metadata, file, indent=2, ensure_ascii=False)

print("Saved artifacts")

Saved artifacts
